# 25B The agent loop
A budget guard, a trace with stated rationales, and replays that are byte-identical. The model here is a deterministic stand-in; yours goes through the record/replay client.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
tools = standin.read_tools(s, audit.Ledger(clock=clock.clock), PANEL, MANDATES, M)
PLAN = [{"tool": "mandate_profile", "args": {"scope": "mandate_a"}, "rationale": "the client's positions"},
        {"tool": "bdc_signals", "args": {"scope": "public"}, "rationale": "what changed this release"},
        {"tool": "bdc_holdings", "args": {"scope": "public"}, "rationale": "terms behind each signal"}]
text, trace = loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "Draft the client note.", clock.clock)
print(pd.DataFrame(trace)[["step", "tool", "action_class", "rationale", "args_hash", "observation_ref", "usd"]].fillna(""))
print("\n" + text)

In [ ]:
forever = standin.FixtureModel([{"tool": "bdc_signals", "args": {"scope": "public"}}] * 50)
_, t1 = loop.run_agent(forever, tools, "loop", clock.clock, max_steps=P["budget"]["max_steps_per_task"])
_, t2 = loop.run_agent(standin.FixtureModel(PLAN, usd=0.30), tools, "pricey", clock.clock, usd_cap=P["budget"]["usd_cap_per_task"])
print("a model that loops:", t1[-1]["stop"], "after", len(t1) - 1, "steps")
print("an expensive one:  ", t2[-1]["stop"], "at step", t2[-1]["step"], f"(spent ${sum(r['usd'] for r in t2):.2f} against a ${P['budget']['usd_cap_per_task']:.2f} cap)")

In [ ]:
# Record once, replay forever: same request, same answer, no cost.
import shutil
cache = ROOT / ".cache" / "nb25_llm"
shutil.rmtree(cache, ignore_errors=True)               # start empty, so the demo reruns the same way
calls = []
fake_provider = lambda req: calls.append(1) or {"text": "Juliett: one markdown, one PIK increase.", "usd": 0.00128}
req = {"model": "small", "messages": [{"role": "user", "content": "Summarize Juliett."}]}
print(llm.ReplayClient(cache, "record", call=fake_provider)(req), "| provider calls:", len(calls))
print(llm.ReplayClient(cache, "replay")(req), "| provider calls:", len(calls))
a = json.dumps(loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "x", clock.clock)[1], sort_keys=True)
b = json.dumps(loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "x", clock.clock)[1], sort_keys=True)
print("two replays give byte-identical traces:", a == b)

**Your turn.** Put your real model behind `ReplayClient` (record mode once, replay in tests and CI), route the small model to triage and materiality, and show one dev task replayed twice with identical traces (Part 2).

### Part 2: the real model (`services/agent/`)
- `model.py`: `ClaudeModel` adapts the loop's step protocol to the Messages API (closed-catalog tools from the server's Pydantic schemas, one tool call per step, thinking blocks passed back unchanged). Every request goes through `ReplayClient`.
- `router.py`: `params.yaml` `models.routes` sends the agent loop to the main model and **triage** and **materiality** to the small one, with structured JSON outputs. Neither can grant authority or drop a rule-flagged signal.
- `run.py`: one dev task, record or replay. Record once (needs `ANTHROPIC_API_KEY`):
```
PYTHONPATH=src:. python -m services.agent.run --task dev-01 --mode record
```
Commit `evals/cassettes/`; tests and CI then replay for free.

In [ ]:
sys.path[:0] = [str(ROOT), str(ROOT / "tests")]
from services.agent import router
from services.agent.model import client_for
from services.agent.run import load_task, run_task
from services.chat.session import load_world
WORLD, MODELS = load_world(ROOT), P["models"]
print(pd.DataFrame([(job, role, MODELS[role]["id"], MODELS[role]["effort"], MODELS[role]["usd_per_mtok_in"], MODELS[role]["usd_per_mtok_out"])
                    for job, role in MODELS["routes"].items()], columns=["job", "route", "model", "effort", "$/M in", "$/M out"]).to_string(index=False))

In [ ]:
task, cache = load_task("dev-01"), ROOT / MODELS["cache_dir"]
try:
    runs = [run_task(task, client_for(MODELS, "replay", cache), WORLD, "priya-fixture-token") for _ in range(2)]
    source = "the real recording in " + MODELS["cache_dir"]
except llm.CacheMiss:
    from test_agent import FakeClaude                  # not recorded yet: show the mechanism with a scripted provider
    tmp = ROOT / ".cache" / "nb25_fake"; shutil.rmtree(tmp, ignore_errors=True)
    run_task(task, client_for(MODELS, "record", tmp, call=FakeClaude()), WORLD, "priya-fixture-token")
    runs = [run_task(task, client_for(MODELS, "replay", tmp), WORLD, "priya-fixture-token") for _ in range(2)]
    source = "a SCRIPTED provider (record dev-01 to replace it with the real model)"
(rec, trace), (_, trace2) = runs
print("replaying", source)
print(pd.DataFrame(trace)[["step", "tool", "action_class", "args_hash", "observation_ref", "tokens_in", "tokens_out", "usd"]].fillna(""))
print("\nactions:", [(a["tool"], a["decision"]) for a in rec["actions"]], "| expected:", task["expect"])
print("two replays give byte-identical traces:", json.dumps(trace, sort_keys=True) == json.dumps(trace2, sort_keys=True))
print("\n" + rec["final_response"])

In [ ]:
import subprocess
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-rs", str(ROOT / "tests" / "test_agent.py")], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-800:] or r.stderr[-800:])
assert r.returncode == 0